In [6]:
import numpy as np
import transforms3d as t3d


class Demo:
    def __init__(self):
        self.relative_translation_scale = 1.0

    def set_start_tcp(self, robot_tcp, unity_tcp):
        self.start_real_tcp = robot_tcp
        self.start_unity_tcp = unity_tcp

    def calc_relative_target(self, pos_from_unity):
        start_real_tcp = self.start_real_tcp
        start_unity_tcp = self.start_unity_tcp

        target = np.zeros(7)

        # --- 平移 ---
        target[:3] = (
            self.relative_translation_scale
            * (pos_from_unity[:3] - start_unity_tcp[:3])
            + start_real_tcp[:3]
        )

        # --- 旋转 ---
        R_unity_current = t3d.quaternions.quat2mat(pos_from_unity[3:])
        print("R_unity_current:\n", R_unity_current)
        print("posfrom_unity quat:\n", pos_from_unity[3:])
        R_unity_start = t3d.quaternions.quat2mat(start_unity_tcp[3:])
        R_robot_start = t3d.quaternions.quat2mat(start_real_tcp[3:])

        target_rot_mat = (
            R_unity_current
            @ np.linalg.inv(R_unity_start)
            @ R_robot_start
        )

        target[3:] = t3d.quaternions.mat2quat(target_rot_mat)
        print("target_rot_mat:\n", target_rot_mat )
        print("target quat:\n", target[3:])
        return target


# =============================
# 测试数据
# =============================

demo = Demo()

# 机器人起始 TCP
robot_start = np.array([
    0.5, 0.0, 0.2,   # 位置
    1.0, 0.0, 0.0, 0.0  # 单位四元数
])

# Unity 起始 TCP
unity_start = np.array([
    1.0, 2.0, 3.0,
    1.0, 0.0, 0.0, 0.0
])

demo.set_start_tcp(robot_start, unity_start)

# 当前 Unity 状态：
# 平移 x +0.1
# 绕 z 轴旋转 90°
angle = np.deg2rad(90)
q_z_90 = [np.cos(angle/2), 0, 0, np.sin(angle/2)]

unity_current = np.array([
    1.1, 2.0, 3.0,
    *q_z_90
])

target = demo.calc_relative_target(unity_current)

print("目标TCP:")
print("位置:", target[:3])
print("四元数:", target[3:])

R_unity_current:
 [[ 2.22044605e-16 -1.00000000e+00  0.00000000e+00]
 [ 1.00000000e+00  2.22044605e-16  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
posfrom_unity quat:
 [0.70710678 0.         0.         0.70710678]
target_rot_mat:
 [[ 2.22044605e-16 -1.00000000e+00  0.00000000e+00]
 [ 1.00000000e+00  2.22044605e-16  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
target quat:
 [0.70710678 0.         0.         0.70710678]
目标TCP:
位置: [0.6 0.  0.2]
四元数: [0.70710678 0.         0.         0.70710678]
